# 証券営業インテリジェンス ハンズオン
## Part 1: ダミーデータ生成

このノートブックでは、ハンズオン用のダミーデータを生成します。
SQL GENERATOR を使用して、100人分の顧客データ・ポートフォリオ・取引履歴などを高速に一括生成します。

> **Note**: このデータはすべて架空のものです。実際の顧客情報は一切含まれていません。

In [ ]:
%%sql -r result_use
USE DATABASE SNOWFINANCE_DB;
USE SCHEMA DEMO_SCHEMA;
USE WAREHOUSE DEMO_WH;
SELECT 'Ready!' AS STATUS;

## Step 1: 顧客マスタデータ投入

まず、山田太郎様（デモの主役）と少数のサンプルデータを手動で投入します。
その後、ストアドプロシージャで残りの顧客データを自動生成します。

In [ ]:
%%sql -r result_c001
-- ============================================================
-- デモシナリオの主役: C001 山田太郎様（68歳, 元上場企業役員）
-- ============================================================
INSERT INTO DIM_CUSTOMER VALUES (
    'C001', '山田太郎', 'ヤマダタロウ', 68, '男性', '1957-04-15',
    '元会社役員', '元トヨタ関連会社', '元代表取締役', '東京都', '千代田区',
    '3000万円以上', 800000000, 200000000, '保守的', '資産保全',
    30, 'プライベートバンク', 'RM001', '鈴木一郎',
    '1995-04-01', '2025-01-10',
    FALSE, FALSE, FALSE,
    '孫の海外留学費用2,000万円が必要。トヨタ株を多数保有。相続対策にも関心あり。',
    CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP()
);

-- 山田様の家族情報
INSERT INTO DIM_FAMILY VALUES
('F001', 'C001', '配偶者', '山田花子', 65, '主婦', TRUE, 1, NULL, CURRENT_TIMESTAMP()),
('F002', 'C001', '長男',   '山田次郎', 40, '会社員', TRUE, 2, NULL, CURRENT_TIMESTAMP()),
('F003', 'C001', '長女',   '田中明子', 38, '医師',   TRUE, 3, NULL, CURRENT_TIMESTAMP()),
('F004', 'C001', '孫（長男の子）', '山田健太', 17, '高校生', FALSE, NULL, '海外留学予定', CURRENT_TIMESTAMP());

-- 山田様のライフイベント
INSERT INTO DIM_LIFE_EVENT VALUES
('E001', 'C001', '教育資金', '孫（山田健太）の米国大学留学費用。4年間で約2,000万円が必要。2025年9月入学予定。', '2025-09-01', 20000000, '高', '検討中', 'F004', '2025-01-10', '2025-01-10', 'トヨタ株売却か、ローン利用か検討中'),
('E002', 'C001', '相続対策', '相続税対策の見直し。現在の資産構成では相続税が高額になる見込み。', '2025-12-31', NULL, '中', '検討中', NULL, '2024-06-01', '2025-01-10', '生前贈与の活用を検討');

SELECT '山田太郎様データ投入完了!' AS STATUS;

In [ ]:
%%sql -r result_portfolio_c001
-- 山田様のポートフォリオ（トヨタ株を大量保有）
INSERT INTO FACT_PORTFOLIO VALUES
('P0001', 'C001', '国内株式', '7203', 'トヨタ自動車', 10000, 1500.00, '2010-04-01', 2850.00, 28500000, 13500000, 90.00, '特定口座', 'JPY', '2025-01-15'),
('P0002', 'C001', '国内株式', '6758', 'ソニーグループ', 5000, 8000.00, '2015-06-01', 15200.00, 76000000, 36000000, 90.00, '特定口座', 'JPY', '2025-01-15'),
('P0003', 'C001', '国内株式', '9433', 'KDDI', 8000, 2800.00, '2012-04-01', 4650.00, 37200000, 14800000, 62.86, '特定口座', 'JPY', '2025-01-15'),
('P0004', 'C001', '国内株式', '8058', '三菱商事', 6000, 2000.00, '2018-04-01', 3200.00, 19200000, 7200000, 60.00, '特定口座', 'JPY', '2025-01-15'),
('P0005', 'C001', '海外株式', 'AAPL', 'Apple Inc.', 500, 120.00, '2019-01-01', 185.00, 13875000, 4875000, 54.17, '特定口座', 'JPY', '2025-01-15'),
('P0006', 'C001', '投資信託', 'INF001', '日経225インデックスファンド', 100000, 18000.00, '2014-07-01', 25000.00, 25000000, 7000000, 38.89, '積立NISA', 'JPY', '2025-01-15'),
('P0007', 'C001', '国内債券', 'BD001', '個人向け国債（変動10年）', 50000000, 1.00, '2020-04-01', 1.00, 50000000, 0, 0.00, '一般口座', 'JPY', '2025-01-15'),
('P0008', 'C001', '現金・預金', 'CASH', '普通預金・定期預金', 1, 100000000, '2025-01-01', 100000000, 100000000, 0, 0.00, '一般口座', 'JPY', '2025-01-15');

SELECT '山田様ポートフォリオ投入完了!' AS STATUS, SUM(MARKET_VALUE) AS 時価総額
FROM FACT_PORTFOLIO WHERE CUSTOMER_ID = 'C001';

## Step 2: SQL GENERATOR で顧客データを一括生成

Snowflake の `TABLE(GENERATOR(ROWCOUNT => N))` を使用して、残り99人分の顧客データを高速に一括生成します。

> **処理内容**: 職業・資産・リスク許容度・RM割り当てなどをランダムに生成

In [ ]:
%%sql -r result_gen_customers
-- SQL GENERATOR で顧客データを一括生成（C002〜C100）
INSERT INTO DIM_CUSTOMER (
    CUSTOMER_ID, CUSTOMER_NAME, AGE, GENDER, OCCUPATION, PREFECTURE,
    ANNUAL_INCOME_BAND, TOTAL_ASSETS, LIQUID_ASSETS, RISK_TOLERANCE,
    INVESTMENT_PURPOSE, SEGMENT, RM_ID, RM_NAME,
    ACCOUNT_OPEN_DATE, LAST_CONTACT_DATE,
    HAS_NISA, HAS_IDECO, HAS_TRUST_ACCOUNT, INVESTMENT_EXPERIENCE_YEARS
)
WITH g AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY SEQ4()) AS rn,
        UNIFORM(0, 7, RANDOM()) AS occ_idx,
        UNIFORM(0, 7, RANDOM()) AS pref_idx,
        UNIFORM(0, 4, RANDOM()) AS purp_idx,
        UNIFORM(0, 3, RANDOM()) AS risk_idx,
        UNIFORM(0, 5, RANDOM()) AS rm_idx,
        UNIFORM(0, 3, RANDOM()) AS seg_idx,
        UNIFORM(45, 85, RANDOM()) AS age_val,
        UNIFORM(0, 1000, RANDOM()) AS gender_r
    FROM TABLE(GENERATOR(ROWCOUNT => 99))
), seg AS (
    SELECT g.*,
        CASE seg_idx WHEN 0 THEN 'プライベートバンク' WHEN 1 THEN 'ゴールド' WHEN 2 THEN 'シルバー' ELSE 'レギュラー' END AS seg_val,
        CASE seg_idx
            WHEN 0 THEN UNIFORM(1000000000, 9999999999, RANDOM())
            WHEN 1 THEN UNIFORM(100000000, 999999999, RANDOM())
            WHEN 2 THEN UNIFORM(10000000, 99999999, RANDOM())
            ELSE UNIFORM(1000000, 9999999, RANDOM())
        END AS ta
    FROM g
)
SELECT
    'C' || LPAD(CAST(rn + 1 AS VARCHAR), 3, '0'),
    '顧客' || CAST(rn + 1 AS VARCHAR) || '号',
    age_val,
    CASE WHEN gender_r > 300 THEN '男性' ELSE '女性' END,
    CASE occ_idx WHEN 0 THEN '経営者' WHEN 1 THEN '医師' WHEN 2 THEN '弁護士' WHEN 3 THEN '公認会計士' WHEN 4 THEN '不動産オーナー' WHEN 5 THEN '資産家' WHEN 6 THEN '元会社役員' ELSE '大学教授' END,
    CASE pref_idx WHEN 0 THEN '東京都' WHEN 1 THEN '神奈川県' WHEN 2 THEN '大阪府' WHEN 3 THEN '愛知県' WHEN 4 THEN '埼玉県' WHEN 5 THEN '千葉県' WHEN 6 THEN '兵庫県' ELSE '福岡県' END,
    '1000万円以上',
    ta,
    CAST(ta * 0.3 AS DECIMAL(18,0)),
    CASE risk_idx WHEN 0 THEN '保守的' WHEN 1 THEN 'やや保守的' WHEN 2 THEN 'やや積極的' ELSE '積極的' END,
    CASE purp_idx WHEN 0 THEN '資産保全' WHEN 1 THEN '資産形成' WHEN 2 THEN '相続対策' WHEN 3 THEN '教育資金' ELSE '老後資金' END,
    seg_val,
    CASE rm_idx WHEN 0 THEN 'RM001' WHEN 1 THEN 'RM002' WHEN 2 THEN 'RM003' WHEN 3 THEN 'RM004' WHEN 4 THEN 'RM005' ELSE 'RM006' END,
    CASE rm_idx WHEN 0 THEN '鈴木一郎' WHEN 1 THEN '田中二郎' WHEN 2 THEN '佐藤三郎' WHEN 3 THEN '高橋四郎' WHEN 4 THEN '伊藤五郎' ELSE '渡辺六郎' END,
    DATEADD('year', -UNIFORM(1, 15, RANDOM()), '2026-01-01'),
    DATEADD('day', -UNIFORM(0, 90, RANDOM()), CURRENT_DATE()),
    CASE WHEN UNIFORM(0, 1, RANDOM()) > 0.5 THEN TRUE ELSE FALSE END,
    CASE WHEN UNIFORM(0, 1, RANDOM()) > 0.7 THEN TRUE ELSE FALSE END,
    CASE WHEN UNIFORM(0, 1, RANDOM()) > 0.6 THEN TRUE ELSE FALSE END,
    UNIFORM(5, 35, RANDOM())
FROM seg;

SELECT COUNT(*) AS 顧客数 FROM DIM_CUSTOMER;

In [ ]:
%%sql -r result_gen_portfolio
-- SQL GENERATOR でポートフォリオデータを一括生成
INSERT INTO FACT_PORTFOLIO (
    PORTFOLIO_ID, CUSTOMER_ID, ASSET_CLASS, SECURITY_CODE, SECURITY_NAME,
    QUANTITY, ACQUISITION_PRICE, ACQUISITION_DATE,
    CURRENT_PRICE, MARKET_VALUE, UNREALIZED_GAIN, GAIN_LOSS_PERCENT,
    ACCOUNT_TYPE, CURRENCY, AS_OF_DATE
)
WITH g AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY SEQ4()) AS rn,
        UNIFORM(2, 100, RANDOM()) AS cust_num,
        MOD(UNIFORM(0, 999, RANDOM()), 8) AS sec_idx,
        UNIFORM(100, 5000, RANDOM()) AS qty,
        UNIFORM(-20, 40, RANDOM()) AS gain_pct
    FROM TABLE(GENERATOR(ROWCOUNT => 300))
), with_sec AS (
    SELECT g.rn,
        'C' || LPAD(CAST(g.cust_num AS VARCHAR), 3, '0') AS customer_id,
        CASE g.sec_idx
            WHEN 0 THEN '国内株式' WHEN 1 THEN '国内株式'
            WHEN 2 THEN '国内株式' WHEN 3 THEN '国内株式'
            WHEN 4 THEN '海外株式' WHEN 5 THEN '海外株式'
            WHEN 6 THEN '投資信託' ELSE '投資信託'
        END AS asset_class,
        CASE g.sec_idx
            WHEN 0 THEN '7203' WHEN 1 THEN '6758'
            WHEN 2 THEN '9433' WHEN 3 THEN '8058'
            WHEN 4 THEN 'AAPL' WHEN 5 THEN 'MSFT'
            WHEN 6 THEN 'INF001' ELSE 'INF002'
        END AS sec_code,
        CASE g.sec_idx
            WHEN 0 THEN 'トヨタ自動車' WHEN 1 THEN 'ソニーグループ'
            WHEN 2 THEN 'KDDI' WHEN 3 THEN '三菱商事'
            WHEN 4 THEN 'Apple Inc.' WHEN 5 THEN 'Microsoft Corp.'
            WHEN 6 THEN '日経225インデックスファンド' ELSE 'グローバル株式インデックスファンド'
        END AS sec_name,
        CASE g.sec_idx
            WHEN 0 THEN 3200 WHEN 1 THEN 17500
            WHEN 2 THEN 4800 WHEN 3 THEN 3600
            WHEN 4 THEN 220 WHEN 5 THEN 460
            WHEN 6 THEN 28000 ELSE 22000
        END AS cur_price,
        g.qty, g.gain_pct
    FROM g
)
SELECT
    'P' || LPAD(CAST(rn + 20 AS VARCHAR), 4, '0'),
    customer_id,
    asset_class, sec_code, sec_name,
    qty,
    CAST(cur_price / (1.0 + gain_pct / 100.0) AS DECIMAL(18, 4)),
    DATEADD('day', -UNIFORM(30, 730, RANDOM()), CURRENT_DATE()),
    cur_price,
    qty * cur_price,
    CAST(qty * cur_price - qty * cur_price / (1.0 + gain_pct / 100.0) AS DECIMAL(18, 0)),
    CAST(gain_pct AS DECIMAL(8, 2)),
    '特定口座', 'JPY',
    CURRENT_DATE()
FROM with_sec;

SELECT COUNT(*) AS ポートフォリオ件数 FROM FACT_PORTFOLIO;

In [ ]:
%%sql -r result_gen_transactions
-- 山田様の取引履歴（10件・手動データ）
INSERT INTO FACT_TRANSACTION VALUES
('T0001','C001','2025-12-01','買付','国内株式','7203','トヨタ自動車',500,3100,1550000,1550,0,1551550,'特定口座','店頭','RM001'),
('T0002','C001','2025-11-15','売却','国内株式','6758','ソニーグループ',200,17200,3440000,3440,688000,2748560,'特定口座','電話','RM001'),
('T0003','C001','2025-10-01','配当','国内株式','7203','トヨタ自動車',10000,45,450000,0,90000,360000,'特定口座','自動','RM001'),
('T0004','C001','2025-09-01','買付','投資信託','INF001','日経225インデックスファンド',1000,27000,27000000,27000,0,27027000,'積立NISA','Web','RM001'),
('T0005','C001','2025-08-01','買付','国内株式','9433','KDDI',1000,4700,4700000,4700,0,4704700,'特定口座','電話','RM001'),
('T0006','C001','2025-07-15','配当','国内株式','9433','KDDI',8000,80,640000,0,128000,512000,'特定口座','自動','RM001'),
('T0007','C001','2025-06-01','買付','海外株式','AAPL','Apple Inc.',100,200,2000000,2000,0,2002000,'特定口座','Web','RM001'),
('T0008','C001','2025-05-01','配当','国内株式','8058','三菱商事',6000,110,660000,0,132000,528000,'特定口座','自動','RM001'),
('T0009','C001','2025-04-01','買付','国内債券','BD001','個人向け国債',50000000,1,50000000,0,0,50000000,'一般口座','店頭','RM001'),
('T0010','C001','2025-03-01','売却','国内株式','7203','トヨタ自動車',300,3050,915000,915,183000,731085,'特定口座','電話','RM001');

-- SQL GENERATOR で取引履歴を一括生成（T0011〜）
INSERT INTO FACT_TRANSACTION (
    TRANSACTION_ID, CUSTOMER_ID, TRANSACTION_DATE, TRANSACTION_TYPE,
    ASSET_CLASS, SECURITY_CODE, SECURITY_NAME,
    QUANTITY, PRICE, AMOUNT, FEE, TAX, NET_AMOUNT,
    ACCOUNT_TYPE, ORDER_CHANNEL, RM_ID
)
WITH g AS (
    SELECT
        ROW_NUMBER() OVER (ORDER BY SEQ4()) AS rn,
        UNIFORM(1, 100, RANDOM()) AS cust_num,
        MOD(UNIFORM(0, 999, RANDOM()), 7) AS sec_idx,
        MOD(UNIFORM(0, 999, RANDOM()), 5) AS txn_idx,
        MOD(UNIFORM(0, 999, RANDOM()), 5) AS ch_idx,
        UNIFORM(100, 5000, RANDOM()) AS qty,
        UNIFORM(1000, 50000, RANDOM()) AS price,
        DATEADD('day', -UNIFORM(0, 365, RANDOM()), CURRENT_DATE()) AS txn_date
    FROM TABLE(GENERATOR(ROWCOUNT => 150))
), calc AS (
    SELECT g.*,
        CASE txn_idx WHEN 0 THEN '買付' WHEN 1 THEN '売却' WHEN 2 THEN '配当' WHEN 3 THEN '利金' ELSE '買付' END AS txn_type,
        CASE sec_idx WHEN 0 THEN '国内株式' WHEN 1 THEN '国内株式' WHEN 2 THEN '国内株式' WHEN 3 THEN '国内株式' WHEN 4 THEN '海外株式' WHEN 5 THEN '投資信託' ELSE '投資信託' END AS asset_class,
        CASE sec_idx WHEN 0 THEN '7203' WHEN 1 THEN '6758' WHEN 2 THEN '9433' WHEN 3 THEN '8058' WHEN 4 THEN 'AAPL' WHEN 5 THEN 'INF001' ELSE 'INF002' END AS sec_code,
        CASE sec_idx WHEN 0 THEN 'トヨタ自動車' WHEN 1 THEN 'ソニーグループ' WHEN 2 THEN 'KDDI' WHEN 3 THEN '三菱商事' WHEN 4 THEN 'Apple Inc.' WHEN 5 THEN '日経225インデックスファンド' ELSE 'グローバル株式インデックスファンド' END AS sec_name,
        CASE ch_idx WHEN 0 THEN '店頭' WHEN 1 THEN '電話' WHEN 2 THEN 'Web' WHEN 3 THEN 'アプリ' ELSE '店頭' END AS channel,
        LEAST(CAST(qty * price AS DECIMAL(18,0)), 100000000) AS amount
    FROM g
)
SELECT
    'T' || LPAD(CAST(rn + 10 AS VARCHAR), 4, '0'),
    'C' || LPAD(CAST(cust_num AS VARCHAR), 3, '0'),
    txn_date, txn_type, asset_class, sec_code, sec_name,
    qty, price, amount,
    CAST(amount * 0.001 AS DECIMAL(18,0)),
    CASE WHEN txn_type = '売却' THEN CAST(amount * 0.02 AS DECIMAL(18,0)) ELSE 0 END,
    CASE WHEN txn_type = '売却' THEN amount - CAST(amount*0.001 AS DECIMAL(18,0)) - CAST(amount*0.02 AS DECIMAL(18,0))
         ELSE amount + CAST(amount*0.001 AS DECIMAL(18,0)) END,
    '特定口座', channel,
    'RM' || LPAD(CAST(MOD(UNIFORM(0,999,RANDOM()), 6) + 1 AS VARCHAR), 3, '0')
FROM calc;

SELECT COUNT(*) AS 取引件数 FROM FACT_TRANSACTION;

## Step 3: 信託銀行商品・ニュース・アナリストレポートの投入

これらのデータは `setup.sql` の Part 4〜5 に含まれています。
以下のセルで `setup.sql` の該当部分を実行してください。

> **Tip**: 大量のINSERT文を実行するため、`setup.sql` ファイルをSnowsightのワークシートで実行することを推奨します。

In [ ]:
%%sql -r result_verify_all
-- データ確認
SELECT TABLE_NAME, ROW_COUNT
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'DEMO_SCHEMA' AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_NAME;

## まとめ

Part 1 完了！ダミーデータの生成が完了しました。

| テーブル | 件数 |
|---|---|
| DIM_CUSTOMER | 100人（C001含む） |
| DIM_FAMILY | 約150件 |
| DIM_LIFE_EVENT | 約50件 |
| FACT_PORTFOLIO | 約300件 |
| FACT_TRANSACTION | 約160件 |

**次のステップ:** `part2_security.ipynb` でCortex AIのセキュリティ設定を体験してください。